# Lean Agent Playground

Wires up a smolagents `CodeAgent` with a hand-picked list of tools, runs it, and saves the run logs to disk.

Requires `OPENAI_API_KEY` in `.env`.

In [1]:
from smolagents import CodeAgent, ToolCallingAgent, OpenAIServerModel

from lean_agent import get_settings, save_run
from lean_agent.tools import (
    caesar_cipher,
    compound_interest,
    fibonacci,
    get_current_time,
    is_palindrome,
    is_prime,
    reverse_words,
    roll_dice,
    temperature_convert,
    word_count,
)

settings = get_settings()
print(f"model:   {settings.model_id}")
print(f"log dir: {settings.log_dir}")

model:   gpt-5.4-nano
log dir: /Users/michaeltheologitis/Code/lean-agent/logs


---

## 1. Build the agent

Three things to notice:

1. **`tools=[...]`** &mdash; an explicit list. Swap tools in/out to change what the agent can do.
2. **`max_steps`** &mdash; hard cap on the number of reasoning iterations. Useful guard against runaway loops.
3. **`instructions`** &mdash; short, high-priority guidance prepended to the system prompt. Keep it tight; it competes with the rest of smolagents' built-in prompt.

In [2]:
model = OpenAIServerModel(
    model_id=settings.model_id,
    api_key=settings.api_key,
    api_base=settings.api_base,  # None -> hosted OpenAI; set to e.g. http://localhost:8000/v1 for vLLM
)

tools = [
    get_current_time,
    word_count,
    fibonacci,
    compound_interest,
    is_prime,
    is_palindrome,
    temperature_convert,
    roll_dice,
    caesar_cipher,
    reverse_words,
]

agent = CodeAgent(
    tools=tools,
    model=model,
    max_steps=12,
    instructions="Prefer using a tool when one fits the question. Be concise.",
)

print(f"{len(tools)} tools wired up:")
for t in tools:
    print(f"  - {t.name}")

10 tools wired up:
  - get_current_time
  - word_count
  - fibonacci
  - compound_interest
  - is_prime
  - is_palindrome
  - temperature_convert
  - roll_dice
  - caesar_cipher
  - reverse_words


---

## 2. Run a prompt that exercises several tools

In [3]:
task = (
    "Answer each of the following using the available tools:\n"
    "1. Is 'A man, a plan, a canal: Panama' a palindrome?\n"
    "2. Is 9973 prime?\n"
    "3. What is 100 degrees Fahrenheit in Celsius?\n"
    "4. What is the 15th Fibonacci number?\n"
    "Return a short numbered summary."
)

answer = agent.run(task)

print("\n--- final answer ---")
print(answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Answer each of the following using the available tools:                                                         │
│ 1. Is 'A man, a plan, a canal: Panama' a palindrome?                                                            │
│ 2. Is 9973 prime?                                                                                               │
│ 3. What is 100 degrees Fahrenheit in Celsius?                                                                   │
│ 4. What is the 15th Fibonacci number?                                                                           │
│ Return a short numbered summary.                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-nano ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  q1_text = "A man, a plan, a canal: Panama"                                                                       
  q1 = is_palindrome(text=q1_text)                                                                                 
  print("q1 palindrome?", q1)                                                                                      
                                                                                                                   
  q2_n = 9973                                                                                                      
  q2 = is_prime(n=q2_n)                                                                                            
  print("q2 prime?", q2)                                                                                           
                                                                                                                   
  q3 = temperature_convert(value=100, from_unit="F", to_unit="C")                                                  
  print("q3 C:", q3)                                                                                               
                                                                                                                   
  q4_index = 15                                                                                                    
  q4 = fibonacci(n=q4_index)                                                                                       
  print("q4 fib(15):", q4)                                                                                         
                                                                                                                   
  summary = f"1. {q1}\n2. {q2}\n3. {q3} °C\n4. {q4}"                                                               
  final_answer(summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
q1 palindrome? True
q2 prime? True
q3 C: 37.77777777777778
q4 fib(15): 610

Final answer: 1. True
2. True
3. 37.77777777777778 °C
4. 610

[Step 1: Duration 3.16 seconds| Input tokens: 2,673 | Output tokens: 269]


--- final answer ---
1. True
2. True
3. 37.77777777777778 °C
4. 610


In [6]:
print(answer)

1. True
2. True
3. 37.77777777777778 °C
4. 610


---

## 3. Save the run logs

`save_run(agent, answer)` writes two files into a fresh per-run directory under `logs/`. Pass `run_id="my-experiment"` if you want to label this run; otherwise it auto-generates one.

- `run.json` &mdash; `{manifest, prompt, answer, logs}` with per-step + total token usage.
- `transcript.yaml` &mdash; sanitized `system / user / assistant / tool-*` linear chat view.

In [5]:
run_dir = save_run(agent, answer)
print(f"logs saved to: {run_dir}")
for f in sorted(run_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

logs saved to: /Users/michaeltheologitis/Code/lean-agent/logs/20260528T010217Z-db9aca
  run.json  (13696 bytes)
  transcript.yaml  (14893 bytes)
